In [6]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv(find_dotenv())

url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host="localhost",
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)

with engine.connect() as con:
    print(con.execute(text("SELECT version()")).scalar())

PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit


In [7]:
DATA = Path("../data")

TABLES = {
    "application_train":     "application_train.csv",
    "bureau":                "bureau.csv",
    "bureau_balance":        "bureau_balance.csv",
    "previous_application":  "previous_application.csv",
    "installments_payments": "installments_payments.csv",
    "credit_card_balance":   "credit_card_balance.csv",
    "pos_cash_balance":      "POS_CASH_balance.csv",
}

ddl = []
for table, fname in TABLES.items():
    df = pd.read_csv(DATA / fname, nrows=200_000)
    df.columns = df.columns.str.lower()
    ddl.append(pd.io.sql.get_schema(df, table, con=engine) + ";")
    print(f"{table}: {df.shape[1]} columns")

Path("sql").mkdir(exist_ok=True)
Path("sql/01_create_tables.sql").write_text("\n\n".join(ddl))
print("\nwritten to sql/01_create_tables.sql")

application_train: 122 columns
bureau: 17 columns
bureau_balance: 3 columns
previous_application: 37 columns
installments_payments: 8 columns
credit_card_balance: 23 columns
pos_cash_balance: 8 columns

written to sql/01_create_tables.sql


In [8]:
from pathlib import Path

Path("sql/02_indexes.sql").write_text("""\
CREATE INDEX IF NOT EXISTS idx_bureau_curr   ON bureau (sk_id_curr);
CREATE INDEX IF NOT EXISTS idx_bureau_bureau ON bureau (sk_id_bureau);
CREATE INDEX IF NOT EXISTS idx_bb_bureau     ON bureau_balance (sk_id_bureau);
CREATE INDEX IF NOT EXISTS idx_prev_curr     ON previous_application (sk_id_curr);
CREATE INDEX IF NOT EXISTS idx_inst_curr     ON installments_payments (sk_id_curr);
CREATE INDEX IF NOT EXISTS idx_cc_curr       ON credit_card_balance (sk_id_curr);
CREATE INDEX IF NOT EXISTS idx_pos_curr      ON pos_cash_balance (sk_id_curr);
ANALYZE;
""")
print("written")

written
